In [ ]:
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv


load_dotenv()

db_llm=ChatOpenAI(model="deepseek-chat",
               base_url=os.getenv("DEEPSEEK_BASE_URL"),
               api_key=os.getenv("DEEPSEEK_API_KEY"),
               temperature=0)

In [ ]:
print(db_llm.invoke("你好，测试连通性").content)

In [ ]:
from langchain_ollama import ChatOllama

coder_llm=ChatOllama(
    base_url=os.getenv("QWEN_URL"),
    model=os.getenv("QWEN_MODEL"),
    num_predict=8192
)

In [ ]:
# from langchain_core.messages import AIMessage

# print(coder_llm.invoke("帮我写一个使用Python实现的贪吃蛇的游戏代码").content)

In [ ]:
# coder_llm_with_deepseek=ChatOpenAI(
#     model="deepseek-coder",
#                base_url=os.getenv("DEEPSEEK_BASE_URL"),
#                api_key=os.getenv("DEEPSEEK_API_KEY"),
#                temperature=0
# )

In [ ]:
# from langchain_core.messages import AIMessage

# print(coder_llm_with_deepseek.invoke("帮我写一个使用Python实现的贪吃蛇的游戏代码").content)

pip install faker

In [ ]:
from sqlalchemy import create_engine, Table, Column, Integer, String, MetaData,ForeignKey,Float
from sqlalchemy.orm import declarative_base, sessionmaker
import pymysql  # ✅ 显式导入，让 SQLAlchemy 识别
from faker import Faker
import random


# 创建基类
Base = declarative_base()

class SalesData(Base):
    __tablename__ = 'sales_data'
    sales_id = Column(Integer, primary_key=True)
    product_id= Column(Integer,ForeignKey('product_information.product_id'))
    employee_id=Column(Integer)
    customer_id=Column(Integer,ForeignKey('customer_information.customer_id'))
    sale_date=Column(String(50))
    quantity=Column(Integer)
    amount=Column(Float)
    discount=Column(Float)
    
class CustomerInformation(Base):
    __tablename__ = 'customer_information'
    customer_id=Column(Integer,primary_key=True)
    customer_name=Column(String(50))
    contact_info=Column(String(50))
    region=Column(String(50))
    customer_type=Column(String(50))
    
class ProductInformation(Base):
    __tablename__ = 'product_information'
    product_id=Column(Integer,primary_key=True)
    product_name=Column(String(50))
    category=Column(String(50))
    unit_price=Column(Float)
    stock_level=Column(Integer)
    
class CompetitorAnalysis(Base):
    __tablename__ = 'competitor_analysis'
    competitor_id=Column(Integer,primary_key=True)
    competitor_name=Column(String(50))
    region=Column(String(50))
    market_share=Column(Float)
    

# ✅ mysql:// → mysql+pymysql://，用 PyMySQL 驱动
DATABASE_URI = os.getenv("DATABASE_URI")
engine = create_engine(DATABASE_URI, echo=True)
Base.metadata.create_all(engine)

In [ ]:
from matplotlib import category


Session = sessionmaker(bind=engine)
session = Session()

fake=Faker()

for _ in range(50):
    customer=CustomerInformation(
        customer_name=fake.name(),
        contact_info=fake.phone_number(),
        region=fake.state(),
        customer_type=random.choice(['Retail','Wholesale'])
    )
    session.add(customer)
    
for _ in range(20):
    product=ProductInformation(
        product_name=fake.word(),
        category=random.choice(["Electronics","Clothing","Furniture","Food","Toys"]),
        unit_price=random.uniform(10.0,1000.0),
        stock_level=random.randint(10,100)
    )
    session.add(product)

for _ in range(10):
    competitor=CompetitorAnalysis(
        competitor_name=fake.company(),
        region=fake.state(),
        market_share=random.uniform(0.01,0.2)
    )
    session.add(competitor)
    
for _ in range(100):
    sale=SalesData(
        product_id=random.randint(1,20),
        employee_id=random.randint(1,10),
        customer_id=random.randint(1,50),
        sale_date=fake.date_between(start_date='-1y',end_date='today').strftime('%Y-%m-%d'),
        quantity=random.randint(1,10),
        amount=random.uniform(50.0,5000.0),
        discount=random.uniform(0.0,0.15)
    )
    session.add(sale)

session.commit()
session.close()


In [ ]:
from pydantic import BaseModel,Field
from langchain_core.tools import tool
from typing import Union,Optional

class AddSaleSchema(BaseModel):
    product_id:int
    employee_id:int
    customer_id:int
    sale_date:str
    quantity:int
    amount:float
    discount:float
    
class DeleteSaleSchema(BaseModel):
    sales_id:int

class UpdateSaleSchema(BaseModel):
    sales_id:int
    quantity:int
    amount:float

class QuerySalesSchema(BaseModel):
    sales_id:int

@tool(args_schema=AddSaleSchema)
def add_sale(product_id,employee_id,customer_id,sale_date,quantity,amount,discount):
    """Add sale record to the database"""
    session=Session()
    try:
        new_sale=SalesData(
            product_id=product_id,
            employee_id=employee_id,
            customer_id=customer_id,
            sale_date=sale_date,
            quantity=quantity,
            amount=amount,
            discount=discount
        )
        session.add(new_sale)
        session.commit()
        return {"messages":["销售记录添加成功。"]}
    except Exception as e:
        return {"messages":[f"添加失败，错误原因：{e}"]}
    finally:
        session.close()

@tool(args_schema=DeleteSaleSchema)
def delete_sale(sales_id):
    """Delete sale record from the database."""
    session=Session()
    try:
        sale_to_delete=session.query(SalesData).filter(SalesData.sales_id == sales_id).first()
        if sale_to_delete:
            session.delete(sale_to_delete)
            session.commit()
            return {"messages":["销售记录删除成功。"]}
        else:
            return {"messages":[f"未找到销售记录ID:{sales_id}"]}
    except Exception as e:
        return {"messages":[f"删除失败，错误原因:{e}"]}
    finally:
        session.close()
        
@tool(args_schema=UpdateSaleSchema)
def update_sale(sales_id,quantity,amount):
    """Update sale record in the database"""
    session=Session()
    try:
        sale_to_update=session.query(SalesData).filter(SalesData.sales_id == sales_id).first()
        if sale_to_update:
            sale_to_update.quantity=quantity
            sale_to_update.amount=amount
            session.commit()
            return {"messages":["销售记录更新成功。"]}
        else:
            return {"messages":[f"未找到销售记录ID:{sales_id}"]}
    except Exception as e:
        return {"messages":[f"更新失败，错误原因:{e}"]}
    finally:
        session.close()

@tool(args_schema=QuerySalesSchema)
def query_sales(sales_id):
    """Query sale record in the database"""
    session=Session()
    try:
        sale_data=session.query(SalesData).filter(SalesData.sales_id == sales_id).first()
        if sale_data:
            return {
                "sales_id":sale_data.sales_id,
                "product_id":sale_data.product_id,
                "employee_id":sale_data.employee_id,
                "customer_id":sale_data.customer_id,
                "sale_date":sale_data.sale_date,
                "quantity":sale_data.quantity,
                "amount":sale_data.amount,
                "discount":sale_data.discount
            }
        else:
            return {"messages":[f"未找到销售记录ID:{sales_id}"]}
    except Exception as e:
        return {"messages":[f"查询失败，错误原因:{e}"]}
    finally:
        session.close()

In [ ]:
from typing import Annotated
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL
import json

repl=PythonREPL()

@tool
def python_repl(
    code:Annotated[str,"The python code to execute to generate your chart."],
):
    """Use this to execute python code. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""
    try:
        result=repl.run(code)
    except BaseException as e:
        return f"Failed to execute.Error: {repr(e)}"
    result_str=f"Successfully executed:\n\`\`\` python\n{code}\n\`\`\`\nStdout: {result}"
    return (
        result_str+"\n\nIf you have completed all tasks, respond with FINAL_ANSWER."
    )

In [ ]:
from langgraph.prebuilt import ToolNode
 
tools=[add_sale,delete_sale,update_sale,query_sales,python_repl]
tool_executor=ToolNode(tools)

In [ ]:
from langchain_core.messages import BaseMessage,HumanMessage,ToolMessage
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

def create_agent(llm,tools,system_message:str):
    """Create an agent."""
    prompt=ChatPromptTemplate.from_messages(
        [(
            "system",
            "You are a helpful AI assistant, collaborating with other assistants,"
            "Use the provided tools to progress towards answering the question."
            "If you are unable to fully answer, that's OK, another assistant with different tools"
            "will help where you left off. Execute what you can to make progress."
            "If you or any of the other assistants have the final answer or deliverable,"
            "prefix your reponse with FINAL_ANSWER so the team knows to stop."
            "You have access to the following tools: {tool_names},\n{system_message}"
        ),
         MessagesPlaceholder(variable_name="messages"),]
    )
    prompt=prompt.partial(system_message=system_message)
    prompt=prompt.partial(tool_names=",".join([tool.name for tool in tools]))
    return prompt | llm.bind_tools(tools)

In [ ]:
db_agent=create_agent(
    db_llm,
    [add_sale,delete_sale,update_sale,query_sales],
    system_message="""You should provide accurate data for the code_generator to use,
    and source code shouldn't be the final answer"""
)

code_agent=create_agent(
    coder_llm,
    [python_repl],
    system_message="Run python code to display diagrams or output execution results"
)

In [ ]:
db_agent

In [ ]:
import functools
from langchain_core.messages import AIMessage

def agent_node(state,agent,name):
    result=agent.invoke(state)
    
    if isinstance(result,ToolMessage):
        pass
    else:
        result=AIMessage(**result.dict(exclude={"type","name"}),name=name)
    return {"messages":[result],"sender":name,}

db_node=functools.partial(agent_node,agent=db_agent,name="db_manager")
code_node=functools.partial(agent_node,agent=code_agent,name="code_generator")

In [ ]:
db_node

In [ ]:
from typing import Literal

from langgraph.graph import END

def router(state):
    messages=state["messages"]
    last_message=messages[-1]
    
    if last_message.tool_calls:
        return "call_tool"
    if"FINAL ANSWER" in last_message.content:
        return END
    return "continue"

In [ ]:
import operator
from typing import Annotated,Sequence
from typing_extensions import TypedDict

class AgentState(TypedDict):
    messages:Annotated[Sequence[BaseMessage],operator.add]
    sender:str

In [ ]:
from langgraph.graph import END,StateGraph

workflow=StateGraph(AgentState)

workflow.add_node("db_manager",db_node)
workflow.add_node("code_generator",code_node)
workflow.add_node("call_tool",tool_executor)

workflow.add_conditional_edges(
    "db_manager",
    router,
    {
        "continue":"code_generator",
        "call_tool":"call_tool",
        END:END
    }
)

workflow.add_conditional_edges(
    "code_generator",
    router,
    {
        "continue":"db_manager",
        "call_tool":"call_tool",
        END:END
    }
)

workflow.add_conditional_edges(
    "call_tool",
    lambda x:x["sender"],
    {
        "db_manager":"db_manager",
        "code_generator":"code_generator"
    }
)

workflow.set_entry_point("db_manager")
graph=workflow.compile()

In [ ]:
from IPython.display import display,Image

display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
for chunk in graph.stream(
    {"messages":[HumanMessage(content="根据sales_id使用折线图显示前5名销售的销售总额")]},
                          {"recursion_limit":50},
                          stream_mode="values"):
    print(chunk)

In [ ]:
for chunk in graph.stream(
    {"messages":[HumanMessage(content="帮我删除销售ID是20 的这名销售信息")]},
    {"rescursion_limit":20},
    stream_mode="values"
):
    print(chunk)

In [ ]:
for chunk in graph.stream(
    {"messages":[HumanMessage(content="帮我根据前10名的销售记录ID，生成对应的销售额柱状图")]},
    {"rescursion_limit":20},
    stream_mode="values"
):
    chunk["messages"][-1].pretty_print()